# FCNN with GloVe Embeddings Sentiment Classifier
## This notebook outlines the application of GloVe embeddings on a Fully Connected Neural Network for building a Movie Sentiment Analyzer 

### Importing the necessary libraries

In [4]:
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords

In [5]:
from numpy import array
from tensorflow.keras.preprocessing.text import one_hot
from tensorflow.keras.preprocessing.sequence import pad_sequences
from keras.models import Sequential
from keras.layers import Dense, Flatten
from keras.layers import Embedding
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer





### Download the dataset from Kaggle
https://www.kaggle.com/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews

In [ ]:
movie_reviews = pd.read_csv("data/IMDB_Dataset.csv")

### Data Exploration

In [ ]:
import tensorflow as tf

print(tf.test.is_gpu_available())

print(tf.config.list_physical_devices('GPU'))

In [ ]:
movie_reviews.head()

In [ ]:
movie_reviews.isnull().values.any()

In [ ]:
movie_reviews.shape

In [ ]:
movie_reviews["review"][3]

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline
sns.countplot(x='sentiment', data=movie_reviews)

### Pre-processing of text
- Removing html tags
- Removing punctutations and numbers
- Removing Multiple spaces
- so on

In [13]:
TAG_RE = re.compile(r'<[^>]+>')

def remove_tags(text):
    return TAG_RE.sub('', text)

In [14]:
def preprocess_text(sen):
    # Removing html tags
    sentence = remove_tags(sen)

    # Remove punctuations and numbers
    sentence = re.sub('[^a-zA-Z]', ' ', sentence)

    # Single character removal
    sentence = re.sub(r"\s+[a-zA-Z]\s+", ' ', sentence)

    # Removing multiple spaces
    sentence = re.sub(r'\s+', ' ', sentence)

    return sentence

In [15]:
X = []
sentences = list(movie_reviews['review'])
for sen in sentences:
    X.append(preprocess_text(sen))

In [ ]:
X[3]

### Creating numerical labels from categorical values

In [17]:
y = movie_reviews['sentiment']

y = np.array(list(map(lambda x: 1 if x=="positive" else 0, y)))

### Split the dataset to train and test

In [18]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

In [ ]:
X_train

### Use Tokenizer to integer encode the documents

In [20]:
tokenizer = Tokenizer(num_words=5000)
tokenizer.fit_on_texts(X_train)

X_train = tokenizer.texts_to_sequences(X_train)
X_test = tokenizer.texts_to_sequences(X_test)

In [ ]:
X_train

### Pad Sequences to create equal-length inputs

In [22]:
vocab_size = len(tokenizer.word_index) + 1

maxlen = 500

X_train = pad_sequences(X_train, padding='post', maxlen=maxlen)
X_test = pad_sequences(X_test, padding='post', maxlen=maxlen)

### Load GloVe embedding into memory

### GloVe Embeddings
- glove.6B.50d.txt (` 175 MB)

### GloVe embeddings is a popular word embedding model where each word is represented as a vector of real numbers.

In [ ]:
from numpy import array
from numpy import asarray
from numpy import zeros

embeddings_dictionary = dict()
glove_file = open('glove.6B.50d.txt', encoding="utf8")

for line in glove_file:
    records = line.split()
    word = records[0]
    vector_dimensions = asarray(records[1:], dtype='float32')
    embeddings_dictionary [word] = vector_dimensions
glove_file.close()

The code loops through each line of the GloVe file.

line.split() splits each line into a list of strings. The first element (records[0]) is the word, and the rest are the vector dimensions.

The vector dimensions (a list of numbers) are converted into a NumPy array using asarray with the dtype='float32' to ensure all values are floats.

The word is then stored as a key in embeddings_dictionary, with its corresponding vector as the value.

In [ ]:
embeddings_dictionary

{'the': array([ 4.1800e-01,  2.4968e-01, ... , -7.8581e-01], dtype=float32)}

'the' is a word, and its corresponding word vector (a 50-dimensional array) is stored as its value. 

These embeddings capture semantic meaning, allowing words with similar meanings to have similar vectors.

Pre-trained GloVe Embedding Sizes: 
- GloVe (Global Vectors for Word Representation) provides pre-trained word embeddings in various sizes, typically with dimensions like 50, 100, 200, or 300.
- The number of dimensions is a design choice made when creating the embeddings and influences the balance between the amount of semantic information captured and computational efficiency.

- 50d embeddings: Compact representation, less computational overhead, but may lose some nuanced information.
- 100d, 200d, 300d embeddings: Capture more semantic detail but require more memory and computation.


### Create Embedding matrix for our Kaggle dataset

In [38]:
embedding_matrix = zeros((vocab_size, 50))
for word, index in tokenizer.word_index.items():
    embedding_vector = embeddings_dictionary.get(word)
    if embedding_vector is not None:
        embedding_matrix[index] = embedding_vector

Embedding Matrix:
-  A matrix where each row represents the vector of a word in your vocabulary.
- It's used in neural networks (especially for text) to convert words into their corresponding numerical representations (embeddings) for training.



### Build the model - Basic Neural Network

In [ ]:
model = Sequential()
embedding_layer = Embedding(vocab_size, 50, weights=[embedding_matrix], input_length=maxlen , trainable=False)
model.add(embedding_layer)

model.add(Flatten())
model.add(Dense(1, activation='sigmoid'))

embedding_layer = Embedding(vocab_size, 50, weights=[embedding_matrix], input_length=maxlen, trainable=False)

- The embedding layer is responsible for turning word indices (which represent words in your vocabulary) into dense word vectors of fixed size.
- vocab_size: The size of the vocabulary (the number of unique words).
- 50: The size of the embeddings (50 dimensions) corresponds to the pre-trained GloVe embeddings you're using.
- weights=[embedding_matrix]: The pre-trained GloVe embeddings are passed to the weights parameter.
- input_length=maxlen: The maximum length of your sequences (the number of words in each review).
- trainable=False: The embeddings are static and won't be updated during training.

### Compile the model

In [ ]:
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['acc'])

print(model.summary())

### Fit the model

In [ ]:
history = model.fit(X_train, y_train, batch_size=128, epochs=50, verbose=1, validation_split=0.2)

### Evaluate Training phase

In [ ]:
score = model.evaluate(X_train, y_train, verbose=1)
score

In [ ]:
print("Train Loss:", score[0])
print("Train Accuracy:", score[1])

### Evaluate Testing phase

In [ ]:
score = model.evaluate(X_test, y_test, verbose=1)

In [ ]:
print("Test Loss:", score[0])
print("Test Accuracy:", score[1])

### Plot the training and testing accuracy and loss

In [ ]:
import matplotlib.pyplot as plt

plt.plot(history.history['acc'])
plt.plot(history.history['val_acc'])

plt.title('model accuracy')
plt.ylabel('accuracy')
plt.xlabel('epoch')
plt.legend(['train','test'], loc='upper left')
plt.show()

plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])

plt.title('model loss')
plt.ylabel('loss')
plt.xlabel('epoch')
plt.legend(['train','test'], loc='upper left')
plt.show()

### What's happening here?